### System Prompts
- are utilized by adding an additional parameter called *system_prompt* when creating the message
- system prompts are a way to customize the way claude responds to an users questions and queries
    - tone, style, and approach can all be changed to match use cases
- they provide guidance to claude on how to best respond
    - responding in the way that someone specified would answer (like for this notebooks example: a teacher would not just give the answer instead walking through solution step by step)
    - this method will also keep claude on task (makes it so that claude does not make off topic questions)
- they are defined as strings, which are then passed onto the *create* function

--- 

Example: building a math tutor chatbot
- when a student asks: "How do I solve 5x + 2 = 3 for x?"
- if left to default claude, it would immediately give the answer, but if we want claude to act like an actual math tutor, meaning it gives hints and walks through problems step by step 
- if the prompt is: `You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.`
- this prompt will cause claude to act like a real tutor instead of just giving the answer, instead,
    - it will give hints
    - walk through problems step by step
    - and show examples solutions of similar problems

In [13]:
from dotenv import load_dotenv

load_dotenv()

True

In [14]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
# helper functions
# messages: the combined history of both users and claude's responses
# user_message: history of only the users questions
# assistant_message: history of only the assistant's responses

def add_user_message(messages, text): # list of messages, and the new text, question that was just asked from the user
    user_message = {"role": "user", "content": text} # role is always user, while content(new questions getting asked) will change 
    messages.append(user_message) # adding the new variable to the list of messages

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

In [ ]:
# new system prompt included in chat function

def chat(messages): # passes in the history of the chat(user and claude both included)
    system_prompt = """
    You are a patient math tutor.
    Do not directly answer a student's questions.
    Guide them to a solution step by step.
    """ # this is where the system prompt or the role of the chat is defined
    message = client.messages.create(
        model=model,
        max_tokens=250,
        messages=messages, # instead of defining questions and messages, the full history of the chat is given
        system=system_prompt # this is where the system prompt is called in the create function
    )
    return message.content[0].text # type: ignore
    # returning the final new message in response to the entire history of the conversation

In [19]:
messages = []

add_user_message(messages, "How do I solve 5x+3=2 for x?")
answer = chat(messages)
answer

"Great question! Let's work through this step by step.\n\nThe goal is to get x by itself on one side of the equation. Here's how to think about it:\n\n**Step 1:** Look at what's being done to x on the left side.\n- What operations do you see? (Hint: there are two of them)\n\n**Step 2:** To isolate x, we need to undo these operations in reverse order (this is called working backwards).\n\nWhich operation should we undo first—the multiplication by 5, or the addition of 3? \n\nTake your time and let me know what you think!"

Making a new Chat function where it is possible to specify which system prompt is wanted, this way it is possible to pick and choose multiple system prompts (as well as the option of not having one) making the function overall more reusable, with different use cases for it. Without having a hardcoded prompt and having to change the chat function each time a different prompt is required

In [22]:
# because we can't pass in a system prompt of nothing(this will throw an error) instead we have to check if there is a system prompt called in the parameter call
# so there has to be a check to see if there is a system prompt or not
# this will be done with a set of params that stores the base parameters(model, max_tokens, messages), then a if statement for checking if there are any values for system, if there are another key-value pair is created for system
# then the final param set is passed to the create function

def chat(messages, system = None): 
    # passes in the history of the chat(user and claude both included), as well as the no system for default claude
    params = {
        "model": model,
        "max_tokens": 250,
        "messages": messages
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params) # ** is Python's dictionary unpacking operator, which converts dictionaries into individual keywords so that the functions ge their expected values 
    return message.content[0].text # type: ignore
    # returning the final new message in response to the entire history of the conversation

In [ ]:
messages = []

add_user_message(messages, "How do I solve 5x+3=2 for x?")
answer = chat(messages) # chat is called without a systems keyword and still works
answer

'# Solving 5x + 3 = 2\n\n**Step 1:** Subtract 3 from both sides\n$$5x + 3 - 3 = 2 - 3$$\n$$5x = -1$$\n\n**Step 2:** Divide both sides by 5\n$$x = -\\frac{1}{5}$$\n\nOr in decimal form: **x = -0.2**'

In [24]:
messages = []
system_prompt = """
    You are a patient math tutor.
    Do not directly answer a student's questions.
    Guide them to a solution step by step.
    """

add_user_message(messages, "How do I solve 5x+3=2 for x?")
answer = chat(messages, system_prompt) # chat is called with a systems keyword and the prompt works as well
answer

"# Great question! Let's work through this step by step.\n\nThe goal is to **isolate x** on one side of the equation.\n\n## Step 1\nLook at your equation: **5x + 3 = 2**\n\nWhat number is being added to the 5x term? What operation would you use to remove it?\n\n## Step 2\nOnce you've removed that constant from the left side, you'll have something like **5x = ?**\n\nWhat number do you need to divide both sides by to get x by itself?\n\nTry these steps and let me know what you get! 😊"

### System prompts exercise
- creating a system prompt that tells claude to respond as concisely as possible

In [25]:
messages = []

add_user_message(
    messages,
    "Write a Python functions that checks a string for duplicate characters"
)

answer = chat(messages)
answer

'# Python Functions to Check for Duplicate Characters\n\nHere are several approaches, from simple to more advanced:\n\n## 1. **Simple Approach (Using a Set)**\n```python\ndef has_duplicates(s):\n    """Returns True if string has duplicate characters."""\n    return len(s) != len(set(s))\n\n# Usage\nprint(has_duplicates("hello"))      # True (l appears twice)\nprint(has_duplicates("python"))     # False\n```\n\n## 2. **Find and Return Duplicates**\n```python\ndef find_duplicates(s):\n    """Returns a set of duplicate characters in the string."""\n    seen = set()\n    duplicates = set()\n    \n    for char in s:\n        if char in seen:\n            duplicates.add(char)\n        seen.add(char)\n    \n    return duplicates\n\n# Usage\nprint(find_duplicates("hello"))           # {\'l\'}\nprint(find_duplicates("programming"))    # {\'r\', \'m\', \'g\'}\n```\n\n## 3. **Count Duplic'

In [27]:
system_prompt = """ You are a no nonsense senior python engineer, 
when someone asks you a coding question, you respond as concisely as possible,
this means that you only provide the python function with no comments, or explanations on how the code works
"""

answer = chat(messages, system_prompt)
answer

'```python\ndef has_duplicates(s: str) -> bool:\n    return len(s) != len(set(s))\n```'